## All about Data Quality and Non-Collision Backgrounds

<CENTER><img src="https://github.com/zlmarshall/notebooks-collection-opendata/blob/DQ_and_NCB/images/ATLASOD.gif?raw=1" style="width:50%"></CENTER>

This notebook uses [ATLAS Open Data](http://opendata.atlas.cern) [2025 beta release](https://opendata.atlas.cern/docs/data/for_education/13TeV25_details) to teach you a bit about the concepts of "Data Quality" and "Non-Collision Backgrounds". It is intended for an education audience and is written to be accessible to a wide range of students.

### What is Data Quality?

Data quality is one of the most important concepts in any data analysis. Imagine asking 10 friends to measure the size of a box. They should all get about the same numbers. They might not get exactly the same numbers, though, and you might want to average all of the measurements together. If it turns out that two of your friends were trying to measure the box during a serious earthquake, then it might be reasonable to ignore their measurements and consider only the other eight. That's a simple example of a measurement rejection for data quality reasons. These sorts of issues come up all the time in real-life data analyses: a single weather station might be broken, and so its data is ignored when producing a weather report or forecast, for example.

For a variety of reasons, not every event recorded with the ATLAS detector is used in our data analysis. When the experiment is running, sometimes things go wrong. These can result in bad data quality, and events that have to be set aside. For example, some parts of the ATLAS detector are only turned on after collisions are taking place in the LHC -- they are disabled until the proton beams in the LHC have been ramped up to full energy and are completely stable, for safety reasons. Any events recorded before these parts of the detector are completely on and ready to provide good data are marked "bad".

Within the ATLAS data, there are special flags used by some detector systems to indicate that an individual event should not be used in a data analysis. The collaboration also produces "Good Runs Lists", which are lists of all the blocks of data that *should* be included in an analysis.

Of course, different analyses are affected by different issues. If you want to measure how loud a car stereo is, you might not care if some of the tires don't have enough air in them. Our collaboration tries to balance the possible number of combinations and the importance of changes: we have only a few good runs lists that are sufficient for most data analyses.

### What are Non-collision Backgrounds?

ATLAS is built to study collisions between protons (and between heavy ions) that happen right in the center of the detector. It does that by carefully measuring all the particles that come flying out of each collision, like a camera. That also leaves it sensitive to a variety of other things that can happen, which are generally described as "Non-collision backgrounds". The most common examples of non-collision backgrounds are:

* [Cosmic rays](https://en.wikipedia.org/wiki/Cosmic_ray). These are usually muons that come from the upper atmosphere and fly through the detector (they also fly through *you*, as you're sitting there, all the time). They look just like muons that come from collisions, except they usually come at funny angles (because they aren't passing straight through the center of the detector), and they often come at strange times (they can arrive when the collider has no protons in it! They can also come *between* bunches of protons).

* Detector noise. Just like static on a radio, the electronics in our detector sometimes have noise. Usually these look different from the signals we are looking for (just like you can tell the difference between static on your radio and music). Sometimes the static is so loud we can't hear the music any more, and then the data have to be discarded.

* Beam halo. As the protons fly around the ring of the LHC in bunches, there are [collimators](https://en.wikipedia.org/wiki/Collimator) that block any protons that get too far out of a normal orbit. Those protons don't just stop; they can produce sprays of particles that run parallel to the original proton beam and sometimes find their way into the detector. They usually come at funny angles, and move across the detector from one side to the other, rather than coming out of the center like particles from a collision.

* Beam gas. The proton bunches go around the LHC in a beam pipe that is *nearly* a vacuum, but it's not perfect. Sometimes there can be collisions between protons from the beam and stray atoms of gas in the beam pipe. They tend to be very asymmetric-looking (because the gas is drifting slowly, and the beam is running full-speed into it), and they tend to be off-center (because the gas can be anywhere in the beam pipe, not only in the very center).

One reason that data quality and non-collision backgrounds are intricately linked is that sometimes there is a period of time that suffers from serious non-collision backgrounds, and therefore must be excluded from our good runs lists.

## Getting into the data

Now that we know the ideas, we're going to get into using the ATLAS Open Data to look at non-collision backgrounds and data quality!

Because these backgrounds and data quality issues are cut out of the data, we don't try to model them in our [detector simulation](https://opendata.atlas.cern/docs/documentation/monte_carlo/introduction_MC). That means we are going to look only into the data for this analysis.

## ATLAS Open Data Initialisation

### First time package installation on your computer (not needed on mybinder)
This first cell installs the required python packages.
It only needs to be run the first time you open this notebook on your computer.
If you close Jupyter and re-open on the same computer, you won't need to run this first cell again.

If this is opened on mybinder, you don't need to run this cell.

In [1]:
import sys
import os.path
!pip install atlasopenmagic
from atlasopenmagic import install_from_environment
install_from_environment()

Installing packages: ['aiohttp>=3.9.5', 'atlasopenmagic>=1.2.0', 'awkward>=2.6.7', 'awkward-pandas>=2023.8.0', 'coffea~=0.7.0', 'fsspec>=2025.7.0', 'hist>=2.8.0', 'ipykernel>=6.29.5', 'jupyter>=1.0.0', 'lmfit>=1.3.2', 'matplotlib>=3.9.1', 'metakernel>=0.30.2', 'notebook<7', 'numpy>=1.26.4', 'pandas>=2.2.2', 'papermill>=2.6.0', 'pip>=24.2', 'scikit-learn>=1.5.1', 'uproot>=5.3.10', 'uproot3>=3.14.4', 'fsspec-xrootd>=0.5.1', 'jupyterlab_latex~=3.1.0', 'vector>=1.4.1']
Installation complete. You may need to restart your Python environment for changes to take effect.


We're going to import a number of packages to help us:
* `numpy`: provides numerical calculations such as histogramming
* `matplotlib`: common tool for making plots, figures, images, visualisations
* `uproot`: processes `.root` files typically used in particle physics into data formats used in python
* `awkward`: introduces `awkward` arrays, a format that generalizes `numpy` to nested data with possibly variable length lists
* `vector`: to allow vectorized 4-momentum calculations

In case this fails the first time you run it, and you've just installed new packages, you might just need to restart the jupyter kernel and try again.

In [2]:
import numpy as np # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
import matplotlib_inline # to edit the inline plot format
#matplotlib_inline.backend_inline.set_matplotlib_formats('pdf', 'svg') # to make plots in pdf (vector) format
from matplotlib.ticker import AutoMinorLocator # for minor ticks
import uproot # for reading .root files
import awkward as ak # to represent nested data in columnar format
import hist # for histogramming
# Import vector and set it up for use with awkward array
import vector
vector.register_awkward()

We will use the [atlasopenmagic](https://opendata.atlas.cern/docs/data/atlasopenmagic) to access the open data directly from the ATLAS OpenData Portal so no need to download any samples. First we need to set the right release.

In [3]:
import atlasopenmagic as atom
atom.available_releases()
atom.set_release('2025e-13tev-beta')

Available releases:
2016e-8tev        2016 Open Data for education release of 8 TeV proton-proton collisions (https://opendata.cern.ch/record/3860).
2020e-13tev       2020 Open Data for education release of 13 TeV proton-proton collisions (https://cern.ch/2r7xt).
2024r-pp          2024 Open Data for research release for proton-proton collisions (https://opendata.cern.record/80020).
2024r-hi          2024 Open Data for research release for heavy-ion collisions (https://opendata.cern.ch/record/80035).
2025e-13tev-beta  2025 Open Data for education and outreach beta release for 13 TeV proton-proton collisions(https://opendata.cern.ch/record/93910).
2025r-evgen       2025 Open Data for research release for event generation (https://opendata.cern.ch/record/160000).
Fetching and caching all metadata for release: 2025e-13tev-beta...
Successfully cached 374 datasets.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


## Example 1: Bad jets

The Monte Carlo simulation is free from non-collision backgrounds and data quality issues, so we will look directly at the detector data for examples.

In [4]:
# Build a list of the data samples to go over; we'll just get the locations of the files we need
data_samples = atom.get_urls('data', protocol='https')

One give-away that something has gone wrong in an event (that a non-collision background or some other data quality issue is present) is a large amount of missing transverse momentum. In the LHC, the proton beams collide head-on. Because of momentum conservation, if there are particles that fly out in one direction, some particles have to fly out in the opposite direction. An imbalance in the momentum in the collision might be an indication of detector signals that aren't coming from a collision. Let's select events with no leptons (for the more advanced student: this will help remove Standard Model events that often have missing transverse momentum from neutrinos, like the decays of W bosons into a lepton and a neutrino), at least one jet above 200 GeV, and at least 100 GeV of missing transverse momentum.

In [5]:
# Define what variables are important to our analysis
# We want the number of leptons, number of jets, the transverse momenta and phi angles of the jets, and the amount of missing transverse momentum
variables = ['lep_n','jet_n','jet_pt','jet_phi','met']

Now we can run over all the data and see what events pass our selections

In [ ]:
# We'll use the hist package to define a histogram in advance
# We'll be filling it as we go along, so that we save minimal data in memory
phihist = hist.Hist(hist.axis.Regular(40, -np.pi, np.pi, label="Leading jet phi"))

# In case you'd like to run over a subset of the data, specify a number of events here
# -1 will not stop; any positive number will stop once the histogram has at least that
# many entries in it
stop_early = 10000

for afile in data_samples:
    # Tell the folks how we are doing
    print(f'Working on file {afile}')

    # Open the tree called "analysis" from our file - that's where the event data are
    tree = uproot.open({'simplecache::'+ afile:"analysis"})

    # Perform the cuts for each data entry in the tree
    # the data will be in the form of an awkward array because we are using the library 'ak' (for awkward)
    for n,data in enumerate(tree.iterate(variables, library="ak")):
        # Let's let folks know how things are going regularly, so they don't get worried
        if (n+1)%10==0:
            print(f'Working on data chunk {n+1}')

        # In this array programming setup, cuts are defined like boolean objects
        # They can be added together with &

        # Start with a cut: Require zero leptons in the event
        cut = (data['lep_n']==0)

        # Add a cut: Require at least one jet
        cut = cut & (data['jet_n']>0)

        # Add a cut: Require at least 100 GeV of MET
        cut = cut & (data['met']>100)

        # Add a cut: Require at least one jet with at least 150 GeV of pT
        cut = cut & (data['jet_pt']>150)

        # Plot the phi of the highest pT jet after the above cuts are applied
        # This is tricky syntax! We get the 'jet_phi' values from data;
        # We apply the cut, and we ask for the first element of the array
        # (because the jet variables are sorted by the pT of the jet, this
        # gives us the phi of the highest transverse momentum jet)
        # Then we flatten the data with awkward - that strips out all the
        # events that didn't pass our cuts, and gives us just a list of the
        # phi values for the jets. Finally, we fill the histogram with that
        # list of phi values
        phihist.fill( ak.flatten(data['jet_phi'][cut][:,:1]) )

        # If we were asked to stop early, stop now
        if stop_early>0 and phihist.sum()>stop_early:
            print(f'Stopping early after {phihist.sum()} entries in the histogram (requested at least {stop_early}).')
            break
    # Little trick to continue only if we didn't try to stop early
    else:
        continue
    break

Working on file simplecache::https://opendata.cern.ch/eos/opendata/atlas/rucio/user/egramsta/data15_periodD.noskim.root


We can now plot the data using Matplotlib. Most of the code here is for the aesthetics of the plot.

In [ ]:
# We're going to make a plot with the matplotlib package
# It's quite powerful and has lots of nice features, but the syntax takes some getting used to

# Set up the figure and the axes on which we'll draw
fig, ax = plt.subplots(figsize=(8, 5))

# Convert from our hist histogram to a numpy array of values
y,bin_edges = phihist.to_numpy()
# Calculate errors
# The statistical uncertainty is just the square root of the number of counts
yerr = np.sqrt(y)
# Slightly awkward - convert from bin edges, which is what hist gives us, into bin centers, which is what matplotlib wants
x = [ (bin_edges[n]+bin_edges[n+1])/2. for n in range(len(bin_edges)-1) ]

# Now we can define the actual plot
ax.errorbar(x=x, y=y, yerr=yerr,
                    fmt='ko', # 'k' means black and 'o' is for circles
                    label='Data')

# x-axis label
ax.set_xlabel(r'Leading jet phi',
                    fontsize=13, x=1, horizontalalignment='right')

# write y-axis label for main axes
ax.set_ylabel('Jets',
                     fontsize=13, y=1, horizontalalignment='right')

# set y-axis limits for main axes
ax.set_ylim( bottom=0, top=np.amax(y)*1.2 )

# add minor ticks on y-axis for main axes
ax.yaxis.set_minor_locator( AutoMinorLocator() )

# draw the legend
ax.legend( frameon=False ); # no box around the legend

A great success! Now, what does the plot mean?

What we're showing here is the angle of the highest transverse momentum jet in the event. The detector is designed to be symmetrical, like a can. The proton collisions don't know which way is up. So, if there are no detector issues and no non-collision backgrounds, this should be a perfectly flat distribution. It's not!

There are some small bumps that you can see around -2.5 to -0.5, and 0.5 to 2.5. Those are generally because some parts of the detector had problems. In this case, they weren't severe enough problems that we removed the data, but if a particular data analysis is very sensitive, it might have to apply corrections to fix those issues.

You can see two *big* excesses at 0 and pi (the edges of the histogram). Those are because of non-collision backgrounds! The way the collimators are arranged, the shielding of the detector, and even the physical structures supporting it (which are often made of steel) block more of the non-collision background at some angles than at others, and those bumps show areas where there's additional background created that doesn't get blocked.

For some data analyses, this is very dangerous! For example, if someone is searching for new physics that leaves a signal in the detector like a high-momentum jet and missing transverse momentum (that's what dark matter would look like in our detector!), then they need to be able to cut down on the amount of non-collision background. We have various tricks to do that (some of which are called "jet cleaning", because they are built to identify jets that have particularly funny properties in the detector) that the most sensitive analyses have to carefully apply.

Feel free to play around with the jet transverse momentum and missing transverse momentum cuts to see how you can enhance the non-collision background in the plot. If you don't require any missing transverse momentum, the background is totally invisible!

## Example 2: Data Quality

Our Open Data for Education and Outreach only has data that is included in good run lists, to avoid any students accidentally looking at bad data. We will need to swap over to using the Open Data for Research in order to look at data outside the good runs lists.

In [ ]:
# Change to the 2024 Research Open Data release
atom.set_release('2024r-pp')
# Get a new set of files
data_samples = atom.get_urls('data', protocol='https')

Now we're going to build a very simple analysis that looks at jet transverse momentum in events that ATLAS thinks are "good for analysis" and events that ATLAS thinks are "not good for analysis". If you've never seen jets before, all you really need to know is that they are sprays of particles in the detector, and they're pretty important for a lot of physics that we do in ATLAS.

First, we're going to get the standard ATLAS Good Runs Lists. This is a little clunkier than it would be in a real analyses because we're doing things in a way that doesn't require any special tools - only python. The good runs lists are in XML format and they're available on the web, so we can grab them from there and reformat the data into something we can use later on.

Data taking in ATLAS is divided into "runs", which usually last about 10 hours. Normally one run corresponds to one time that the LHC is filled with bunches of protons. Those bunches circulate for around 10 hours, colliding constantly, and then the beams are ejected and a new set of bunches are injected. During the collisions, ATLAS takes data. The data are divided into luminosity blocks, which are about one minute of continuous data. Those luminosity blocks are usually expected to have consistent conditions internally, so we keep or reject all the data of a luminosity block in a good runs list. This is all represented in a good runs list in the form of "LumiBlockCollections". Each LumiBlockCollection includes all the ranges of luminosity blocks that should be considered "good" within a single run.

In [ ]:
# Tools for getting webpage data and parsing XML
import requests
import xml.etree.ElementTree as ET
# Get the 2015 and 2016 good runs lists
grl2015 = ET.fromstring(requests.get('https://atlas-groupdata.web.cern.ch/atlas-groupdata/GoodRunsLists/data15_13TeV/20170619/data15_13TeV.periodAllYear_DetStatus-v89-pro21-02_Unknown_PHYS_StandardGRL_All_Good_25ns.xml').text)
grl2016 = ET.fromstring(requests.get('https://atlas-groupdata.web.cern.ch/atlas-groupdata/GoodRunsLists/data16_13TeV/20180129/data16_13TeV.periodAllYear_DetStatus-v89-pro21-01_DQDefects-00-02-04_PHYS_StandardGRL_All_Good_25ns.xml').text)

# Parse the good runs lists into a dictionary of runs and luminosity block ranges
LBs = {}
for c in grl2015.iter('LumiBlockCollection'):
    my_run = int(c.find('Run').text)
    LBs[my_run]=[]
    for lb in c.iter('LBRange'):
        LBs[my_run] += [ [ int(lb.attrib['Start']) , int(lb.attrib['End']) ] ]
for c in grl2016.iter('LumiBlockCollection'):
    my_run = int(c.find('Run').text)
    LBs[my_run]=[]
    for lb in c.iter('LBRange'):
        LBs[my_run] += [ [ int(lb.attrib['Start']) , int(lb.attrib['End']) ] ]

print(f'Runs included in the good runs list: {LBs.keys()}')

To use the ranges of luminosity blocks that we just defined, we're going to want a function that checks to see if the event is inside or outside of the good runs list.

This function is also going to take a "cleaning variable" called `DFCommonJets_eventClean_LooseBad`. There's an algorithm that runs when ATLAS makes its analysis data formats (formats called "derivations", and "DF" is for "Derivation Framework") that looks for specific issues that come up by examining the jets in the event. For example, jets tend to have more energy in the middle than they do at the edges when they come from particle collisions. In some events, there might be a whole lot of noise in the calorimeter that results in our seeing enough energy to think it's a jet, but because it's just noise the jet would look much "wider" (more energy at the edges) than normal. Similarly, we have a pretty good understanding of what fraction of a jet's energy will be deposited in each part of our detector. If there is a whole lot of energy in just one detector layer, that might mean that single layer had too much noise in that particular event. When we see these noise bursts we usually remove the entire event, just to be safe.

In case you're already thinking ahead: yes, when we do searches for strange-looking new particles we have to be _extremely_ careful that the selection criteria we use to remove events with noise don't accidentally also remove all the events with new particles! There's a lot of work that goes into that in some analyses. For now, we'll assume the standard variable is doing a reasonably good job.

In [ ]:
# Simple helper for applying the good runs list luminosity block selection
def in_GRL_and_clean( run , lumiblock , clean ):
    # First we'll see if the event passed our "cleaning selection"
    if not clean:
        return False
    # If the run is not in our good runs list, fail
    if run not in LBs:
        return False
    # Loop through all the luminosity block ranges for the run
    for lb_pair in LBs[run]:
        # If the luminosity block is in a good range, succeed
        if lb_pair[0]<=luminblock<=lb_pair[1]:
            return True
    # Didn't find the lumiblock in a good range; fail
    return False

Just like we did last time, we're going to set up a histogram and fill it as we go. We'll want a different set of variables to look at for this setup as well, because the data are in a different format. Handling the PHYSLITE data takes a little more care because of its structure, so we're going to very carefully follow the patterns used in the [PHYSLITE tutorial](https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/for-research/physlite_tutorial.ipynb) for the research open data. We'll try to include enough comments that someone who has never seen this before is able to follow what the code is doing.

In [ ]:
# We'll use the hist package to define a histogram in advance
# We'll be filling it as we go along, so that we save minimal data in memory
grlhist = hist.Hist(hist.axis.Regular(100, 200, 1500, label="Jet p_{T} [GeV] (GRL)"))
nogrlhist = hist.Hist(hist.axis.Regular(100, 200, 1500, label="Jet p_{T} [GeV] (no GRL)"))

# In case you'd like to run over a subset of the data, specify a number of events here
# -1 will not stop; any positive number will stop once the histogram has at least that
# many entries in it
stop_early = 1000

for afile in data_samples:
    # Tell the folks how we are doing
    print(f'Working on file {afile}')

    # The tree with event data in the PHYSLITE files is called "CollectionTree", rather than "analysis"
    tree = uproot.open({'simplecache::'+ afile:'CollectionTree'})

    # Perform the cuts for each data entry in the tree
    # the data will be in the form of an awkward array because we are using the library 'ak' (for awkward)
    for n,data in enumerate(tree.iterate(variables, library="ak")):
        # Let's let folks know how things are going regularly, so they don't get worried
        if (n+1)%10==0:
            print(f'Working on data chunk {n+1}')

        # Now we're going to make two different cuts on the data.
        # First we'll ask that events be in the good runs list and pass our
        # standard cleaning requirements. Our variables get kinda funny names;
        # check out the PHYSLITE tutorial to understand the variables better
        grl_data = data[in_GRL_and_clean(data['EventInfoAuxDyn.runNumber'],
                                         data['EventInfoAuxDyn.lumiBlock'],
                                         data['EventInfoAuxDyn.DFCommonJets_eventClean_LooseBad'])]
        grlhist.fill( ak.flatten(grl_data['AnalysisJetsAuxDyn.pt'][:,:1]) )

        # Next we'll do the same, but for data _not_ in the GRL or not clean
        no_grl_data = data[~in_GRL(data['EventInfoAuxDyn.runNumber'],
                                   data['EventInfoAuxDyn.lumiBlock'],
                                   data['EventInfoAuxDyn.DFCommonJets_eventClean_LooseBad'])]
        nogrlhist.fill( ak.flatten(no_grl_data['AnalysisJetsAuxDyn.pt'][:,:1]) )

        if stop_early>0 and grlhist.sum()>stop_early and nogrlhist.sum()>stop_early:
            print(f'Stopping early after {grlhist.sum()} entries ({nogrlhist.sum()} entries) in the "with GRL" ("no GRL") histogram (requested at least {stop_early}).')
            break
    # Little trick to continue only if we didn't try to stop early
    else:
        continue
    break

Now again we can draw the results with matplotlib. Let's see how our Z-bosons look!

In [ ]:
# We're going to make a plot with the matplotlib package
# It's quite powerful and has lots of nice features, but the syntax takes some getting used to

# Set up the figure and the axes on which we'll draw
fig, ax = plt.subplots(figsize=(8, 5))

# Convert from our hist histogram to a numpy array of values
grl_y,grl_bin_edges = grlhist.to_numpy()
# Calculate errors
# The statistical uncertainty is just the square root of the number of counts
grl_yerr = np.sqrt(grl_y)
# Slightly awkward - convert from bin edges, which is what hist gives us, into bin centers, which is what matplotlib wants
grl_x = [ (grl_bin_edges[n]+grl_bin_edges[n+1])/2. for n in range(len(grl_bin_edges)-1) ]

# Do the same for the no-GRL case
nogrl_y,nogrl_bin_edges = nogrlhist.to_numpy()
# Calculate errors
# The statistical uncertainty is just the square root of the number of counts
nogrl_yerr = np.sqrt(nogrl_y)
# Slightly awkward - convert from bin edges, which is what hist gives us, into bin centers, which is what matplotlib wants
nogrl_x = [ (nogrl_bin_edges[n]+nogrl_bin_edges[n+1])/2. for n in range(len(nogrl_bin_edges)-1) ]

# Now we can define the actual plot
ax.errorbar(x=grl_x, y=grl_y, yerr=grl_yerr,
                    fmt='ko', # 'k' means black and 'o' is for circles
                    label='Data (GRL)')

# And the same for the one without the GRL applied
ax.errorbar(x=nogrl_x, y=nogrl_y, yerr=nogrl_yerr,
                    fmt='ko', # 'k' means black and 'o' is for circles
                    label='Data (No GRL)')

# x-axis label
ax.set_xlabel(r'Jet p_{T} [GeV]',
                    fontsize=13, x=1, horizontalalignment='right')

# write y-axis label for main axes
ax.set_ylabel('Events',
                     fontsize=13, y=1, horizontalalignment='right')

# set y-axis limits for main axes
ax.set_ylim( bottom=0, top=np.amax(y)*1.2 )

# add minor ticks on y-axis for main axes
ax.yaxis.set_minor_locator( AutoMinorLocator() )

# draw the legend
ax.legend( frameon=False ); # no box around the legend